# Web Traffic Forecasting
**Project 7 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **daily Wikipedia article page views** using the **Web Traffic Time Series Forecasting** dataset from the 2017 Kaggle competition. The dataset contains daily view counts for ~145,000 Wikipedia pages across 7 language editions.

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting — Panel (multi-series) |
| **Target variable** | `daily_views` (page views per day) |
| **Granularity** | Daily → aggregated to weekly for modelling |
| **Frequency** | Weekly (`W`) |
| **Primary TS library** | StatsForecast (AutoARIMA, AutoETS, AutoTheta, SeasonalNaive) |
| **Kaggle dataset** | `competitions/web-traffic-time-series-forecasting` |
| **Date range** | July 2015 – September 2017 (~550 days, ~79 weeks) |

We focus on a **representative sample** of top English-language Wikipedia articles, demonstrating how to handle multi-series forecasting with diverse traffic patterns (news spikes, seasonal topics, evergreen content).

## Learning Objectives

1. **Handle wide-format time series** — the competition data is stored with pages as rows and dates as columns; learn to reshape this into a tidy long-format panel
2. **Segment page categories** from page names (article vs. talk page, language edition)
3. **Model traffic spike patterns** caused by news events using residual analysis
4. **Apply StatsForecast** to a sample of diverse page-level series
5. **Aggregate-level vs. series-level forecasting**: when to forecast individual pages vs. aggregated total traffic
6. **Evaluate SMAPE** (Symmetric MAPE) — the competition's official metric
7. **Understand web traffic seasonality**: weekly patterns (lower on weekends for work-related topics) vs. annual patterns

## Problem Statement

Given 550 days of daily Wikipedia page view counts, **forecast the next 60 days** for each page. The original competition forecasted days 901–990, but in this notebook we hold out the last 8 weeks (56 days) for evaluation.

In practice, this maps to: **predict which Wikipedia pages will have high traffic over the next 2 months**, which helps Wikimedia allocate CDN capacity and plan infrastructure for anticipated traffic spikes (e.g., elections, sports events, film releases).

## Why This Project Matters

- **CDN capacity planning**: Wikimedia serves 1B+ monthly unique visitors; traffic forecasts determine edge server allocation
- **Content recommendation**: high predicted traffic = strong candidate for featured article or newsletter inclusion
- **Trend detection**: early traffic growth signals that a topic is gaining public attention
- **Infrastructure cost forecasting**: cloud auto-scaling budgets depend on accurate long-horizon traffic predictions

Beyond Wikipedia, these techniques apply to any URL-level traffic forecasting (news sites, SaaS dashboards, API rate planning).

## Dataset Overview

| File | Description |
|------|-------------|
| `train_1.csv` | ~145K rows × 550 columns: `Page` identifier + daily view count columns (`2015-07-01` ... `2016-12-31`) |
| `train_2.csv` | Same pages, extended through `2017-09-01` |
| `key_1.csv` / `key_2.csv` | Lookup: `Page` → `Id` format for submission |

### Page name format
`Article_name_language.wikipedia.org_access_agent`

For example: `Special:Search_en.wikipedia.org_all-access_all-agents`

We parse this to extract:
- `article`: the canonical article name
- `language`: `en`, `de`, `fr`, `ja`, `zh`, `ru`, `es`
- `access`: `all-access`, `desktop`, `mobile-web`
- `agent`: `all-agents`, `spider`, `user`

### Sampling strategy
We work with the **top 100 English-language `all-access` `all-agents` article pages** by mean daily traffic.

## Dataset Source & License Notes

- **Kaggle competition**: [Web Traffic Time Series Forecasting](https://www.kaggle.com/competitions/web-traffic-time-series-forecasting)
- **Data provider**: Wikimedia Foundation
- **License**: Competition rules; underlying Wikipedia data is CC BY-SA 4.0
- **Note**: Must accept competition rules on Kaggle before downloading

## Environment Setup

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pip in {
    "kagglehub": "kagglehub", "pandas": "pandas", "numpy": "numpy",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "plotly": "plotly",
    "statsforecast": "statsforecast", "statsmodels": "statsmodels",
    "scikit_learn": "scikit-learn", "lazypredict": "lazypredict",
    "flaml": "flaml",
}.items():
    try: __import__(imp)
    except ImportError: _pip(pip)
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib, re
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA, AutoETS, AutoTheta, SeasonalNaive,
    Naive, WindowAverage
)

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration & Constants

In [ ]:
PROJECT_NAME  = "Web Traffic Forecasting"
KAGGLE_COMP   = "web-traffic-time-series-forecasting"
TARGET_COL    = "daily_views"
FREQ          = "W"
HORIZON       = 8               # Forecast 8 weeks ahead
SEASON_LEN    = 52              # Annual weekly seasonality
TEST_WEEKS    = 8
N_TOP_PAGES   = 100             # Number of top EN pages to model
FLAML_BUDGET  = 90
RANDOM_STATE  = 42

print(f"Project : {PROJECT_NAME}")
print(f"Dataset : {KAGGLE_COMP}")
print(f"Horizon : {HORIZON} weeks | Sample: top {N_TOP_PAGES} EN pages | Test: {TEST_WEEKS} weeks")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]:
    if os.environ.get(k): print(f"✓ {k} set."); kaggle_ok = True; break
kj = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kj.exists(): print("✓ kaggle.json found."); kaggle_ok = True
if not kaggle_ok:
    print("NO KAGGLE CREDENTIALS — visit https://www.kaggle.com/settings → API → Create New Token")
    print("Also accept competition rules at https://www.kaggle.com/competitions/web-traffic-time-series-forecasting/rules")
else:
    print("Ready to download.")

## Dataset Download & Loading

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_COMP))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub failed: {e}")
    dl_dir = pathlib.Path("data/web_traffic"); dl_dir.mkdir(parents=True, exist_ok=True)
    os.system(f"kaggle competitions download -c {KAGGLE_COMP} -p {dl_dir}")
    import zipfile
    for z in dl_dir.glob("*.zip"):
        with zipfile.ZipFile(z) as zf: zf.extractall(dl_dir)
    data_path = dl_dir

csv_files = {f.name: f for f in data_path.rglob("*.csv")}
print("Files:", list(csv_files.keys()))

In [ ]:
# ── Load train_2 (larger, more recent) or train_1 as fallback ─────
train_file = csv_files.get("train_2.csv", csv_files.get("train_1.csv"))
print(f"Loading: {train_file.name}  ({train_file.stat().st_size/1e6:.0f} MB)")
raw = pd.read_csv(train_file)
print(f"Shape: {raw.shape}  (rows=pages, cols=Page+daily_dates)")
print(f"Date columns: {raw.columns[1]} ... {raw.columns[-1]}")
print(f"Total pages: {len(raw):,}")

## Page Name Parsing

In [ ]:
def parse_page(page_name):
    # Format: Article_language.wikipedia.org_access_agent
    parts = page_name.split("_")
    # Extract suffix: last 3 underscore-delimited fields (lang, access, agent)
    # Language: 2-letter code before '.wikipedia.org'
    m = re.search(r"_([a-z]{2,3})\.wikipedia\.org_([^_]+)_([^_]+)$", page_name)
    if m:
        lang   = m.group(1)
        access = m.group(2)
        agent  = m.group(3)
        suffix = m.group(0)
        article = page_name[: -len(suffix)]
    else:
        lang = access = agent = "unknown"
        article = page_name
    return article, lang, access, agent

raw["article"], raw["language"], raw["access"], raw["agent"] = zip(
    *raw["Page"].map(parse_page)
)

print("Language distribution:")
print(raw["language"].value_counts().head(10).to_string())
print()
print("Access types:")
print(raw["access"].value_counts().to_string())
print()
print("Agent types:")
print(raw["agent"].value_counts().to_string())

## Sample: Top 100 English All-Access All-Agent Pages

In [ ]:
date_cols = [c for c in raw.columns if re.match(r"\d{4}-\d{2}-\d{2}", c)]
print(f"Date columns available: {len(date_cols)}  ({date_cols[0]} to {date_cols[-1]})")

# Filter EN all-access all-agents
en_mask = (raw["language"] == "en") & (raw["access"] == "all-access") & (raw["agent"] == "all-agents")
en_pages = raw[en_mask].copy()
print(f"English all-access all-agents pages: {len(en_pages):,}")

# Select top N by mean daily views (ignoring NaN)
en_pages["mean_views"] = en_pages[date_cols].mean(axis=1)
top_pages = en_pages.nlargest(N_TOP_PAGES, "mean_views")

print(f"\nTop {N_TOP_PAGES} pages selected by mean daily views")
print(f"Mean views range: {top_pages['mean_views'].min():.0f} – {top_pages['mean_views'].max():.0f}")
print()
print("Top 10 articles:")
print(top_pages[["article", "mean_views"]].head(10).to_string(index=False))

## Reshape to Long Format Panel

In [ ]:
# Wide → Long
long_df = top_pages[["Page", "article"] + date_cols].melt(
    id_vars=["Page", "article"],
    var_name="date",
    value_name="daily_views"
)
long_df["date"]  = pd.to_datetime(long_df["date"])
long_df.sort_values(["Page", "date"], inplace=True)
long_df.reset_index(drop=True, inplace=True)

# Fill NaN (sporadic missing days) with 0
long_df["daily_views"] = long_df["daily_views"].fillna(0).astype(int)

# Aggregate daily → weekly (sum per week)
long_df.set_index("date", inplace=True)
weekly_panel = (long_df.groupby("Page")
                .resample("W")["daily_views"]
                .sum()
                .reset_index()
                .rename(columns={"date": "ds", "daily_views": "y"}))
weekly_panel["unique_id"] = weekly_panel["Page"]

print(f"Long format (daily): {len(long_df):,} rows")
print(f"Weekly panel: {len(weekly_panel):,} rows, {weekly_panel['unique_id'].nunique()} unique pages")
print(f"Weeks per page: {weekly_panel.groupby('unique_id')['ds'].count().describe().to_dict()}")

## Data Validation Checks

In [ ]:
print("=" * 55)
print("DATA VALIDATION — Web Traffic Weekly Panel")
print("=" * 55)

print(f"\nWeeks per series:")
print(weekly_panel.groupby("unique_id")["ds"].count().describe().to_string())

print(f"\nTarget (weekly views) statistics:")
print(weekly_panel["y"].describe().to_string())
print(f"Zeros: {(weekly_panel['y']==0).sum():,} ({(weekly_panel['y']==0).mean()*100:.1f}%)")

print(f"\nDate range: {weekly_panel['ds'].min().date()} to {weekly_panel['ds'].max().date()}")
total_weeks  = weekly_panel["ds"].nunique()
print(f"Total distinct weeks: {total_weeks}")

## Exploratory Data Analysis

In [ ]:
# ── Aggregate total daily traffic ─────────────────────────────────────
total_weekly = weekly_panel.groupby("ds")["y"].sum().reset_index()
fig = px.line(total_weekly, x="ds", y="y",
              title=f"Total Weekly Views — Top {N_TOP_PAGES} English Wikipedia Pages",
              labels={"ds": "Week", "y": "Total Page Views"},
              template="plotly_white")
fig.show()

In [ ]:
# ── Individual series diversity ────────────────────────────────────────
sample_pages = (weekly_panel.groupby("unique_id")["y"].std()
                .nlargest(6).index.tolist())
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, pid in zip(axes.flatten(), sample_pages):
    sub = weekly_panel[weekly_panel["unique_id"] == pid]
    ax.plot(sub["ds"], sub["y"], lw=1)
    article_name = sub["unique_id"].iloc[0].split("_en.wikipedia")[0][:40]
    ax.set_title(f"{article_name}...", fontsize=9)
    ax.set_ylabel("Weekly views")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("Top 6 Most Variable Articles — Individual Traffic Patterns", y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Traffic distribution (log scale) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(weekly_panel["y"], bins=50, color="#2563EB", alpha=0.7, edgecolor="black")
axes[0].set_title("Weekly Views Distribution")
axes[0].set_xlabel("Views")

axes[1].hist(np.log1p(weekly_panel["y"]), bins=50, color="#10B981", alpha=0.7, edgecolor="black")
axes[1].set_title("log1p(Weekly Views) Distribution")
axes[1].set_xlabel("log(1 + Views)")
plt.tight_layout(); plt.show()
print("Log-transform makes distribution more symmetric — models should be trained on log scale.")

In [ ]:
# ── Weekly seasonality (day of week in original daily data) ───────────
daily_dow = long_df.copy()
daily_dow["dow"] = daily_dow.index.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow_means = daily_dow.groupby("dow")["daily_views"].mean().reindex(dow_order)

fig = px.bar(x=dow_means.index, y=dow_means.values,
             title="Average Daily Views by Day of Week (Top 100 EN Pages)",
             labels={"x": "Day of Week", "y": "Avg Daily Views"},
             template="plotly_white", color=dow_means.values,
             color_continuous_scale="Blues")
fig.show()
print(f"Peak day: {dow_means.idxmax()} ({dow_means.max():,.0f} avg views)")
print(f"Low day : {dow_means.idxmin()} ({dow_means.min():,.0f} avg views)")

## Target Analysis

In [ ]:
# Aggregate series for classical analysis
total_ts = total_weekly.set_index("ds")["y"]
print("Aggregate weekly traffic statistics:")
print(f"  Mean   : {total_ts.mean():,.0f}")
print(f"  Std    : {total_ts.std():,.0f}")
print(f"  CV     : {total_ts.std()/total_ts.mean()*100:.1f}%")
print()

# Autocorrelation on aggregate series
from pandas.plotting import autocorrelation_plot
fig, ax = plt.subplots(figsize=(14, 4))
autocorrelation_plot(total_ts, ax=ax)
ax.set_title("Autocorrelation — Aggregate Weekly Web Traffic")
ax.set_xlim(0, min(56, len(total_ts)//2))
plt.tight_layout(); plt.show()

print("ACF at key lags:")
for lag in [1, 2, 4, 7, 13, 26, 52]:
    if lag < len(total_ts)//2:
        print(f"  Lag {lag:>3}: {total_ts.autocorr(lag):.3f}")

## SMAPE — The Competition Metric

$$\text{SMAPE} = \frac{1}{n} \sum_{t=1}^{n} \frac{|y_t - \hat{y}_t|}{(|y_t| + |\hat{y}_t|) / 2} \times 100$$

SMAPE is bounded between 0% and 200% and is **symmetric** — it penalises over-prediction and under-prediction equally. It's preferred over MAPE for web traffic because:
1. Series can temporarily drop to near-zero (after a news event fades)
2. Over-prediction (allocating too much CDN) and under-prediction (page becomes slow) have similar costs

In [ ]:
def smape(actual, predicted):
    a = np.array(actual, dtype=float).flatten()
    p = np.array(predicted, dtype=float).flatten()
    n = min(len(a), len(p))
    a, p = np.abs(a[:n]), np.abs(p[:n])
    denom = (a + p) / 2
    mask = denom > 0
    return np.mean(np.abs(a[mask] - p[mask]) / denom[mask]) * 100

def evaluate(actual, predicted, name):
    a = np.array(actual, float).flatten()
    p = np.array(predicted, float).flatten()
    n = min(len(a), len(p))
    mae_v   = mean_absolute_error(a[:n], p[:n])
    rmse_v  = np.sqrt(mean_squared_error(a[:n], p[:n]))
    smape_v = smape(a, p)
    print(f"  {name:<40s}  MAE:{mae_v:>9.0f}  RMSE:{rmse_v:>9.0f}  SMAPE:{smape_v:>6.1f}%")
    return {"model": name, "MAE": mae_v, "RMSE": rmse_v, "SMAPE": smape_v}

results = []

## Train / Validation / Test Split

In [ ]:
all_weeks = sorted(weekly_panel["ds"].unique())
n_weeks   = len(all_weeks)
test_weeks_list  = all_weeks[-TEST_WEEKS:]
train_weeks_list = all_weeks[:-TEST_WEEKS]

wk_train = weekly_panel[weekly_panel["ds"].isin(train_weeks_list)].copy()
wk_test  = weekly_panel[weekly_panel["ds"].isin(test_weeks_list)].copy()

# Aggregate for classical models
agg_train = total_weekly[total_weekly["ds"].isin(train_weeks_list)].set_index("ds")["y"]
agg_test  = total_weekly[total_weekly["ds"].isin(test_weeks_list)]["y"].values

print(f"Training weeks : {len(train_weeks_list)} ({train_weeks_list[0].date()} to {train_weeks_list[-1].date()})")
print(f"Test weeks     : {len(test_weeks_list)}  ({test_weeks_list[0].date()} to {test_weeks_list[-1].date()})")
print(f"Train series rows: {len(wk_train):,}  |  Test: {len(wk_test):,}")

## Baselines on Aggregate Traffic

In [ ]:
print("BASELINES (aggregate, test = last 8 weeks):")

# Seasonal naive
sn_period = min(52, len(agg_train))
sn_pred = agg_train.values[-(sn_period - np.arange(TEST_WEEKS) % sn_period)]
results.append(evaluate(agg_test, sn_pred, "Seasonal Naive (52w)"))

# 4-week average
results.append(evaluate(agg_test, np.full(TEST_WEEKS, agg_train.values[-4:].mean()), "4-Week Moving Average"))

# Last value
results.append(evaluate(agg_test, np.full(TEST_WEEKS, agg_train.values[-1]), "Naive (last value)"))

## Feature Engineering for Tabular Models

In [ ]:
# Aggregate series features
agg_all = total_weekly.copy()
for lag in [1, 2, 4, 8]:
    agg_all[f"lag_{lag}w"] = agg_all["y"].shift(lag)
agg_all["roll_4w"] = agg_all["y"].shift(1).rolling(4).mean()
agg_all["roll_8w"] = agg_all["y"].shift(1).rolling(8).mean()
agg_all["month"]   = agg_all["ds"].dt.month
agg_all["quarter"] = agg_all["ds"].dt.quarter

FEAT_COLS = [c for c in agg_all.columns if c not in ("ds", "y")]
feat_tr = agg_all[agg_all["ds"].isin(train_weeks_list)].dropna()
feat_te = agg_all[agg_all["ds"].isin(test_weeks_list)].dropna()
print(f"Features: {FEAT_COLS}")
print(f"Train: {len(feat_tr)} rows | Test: {len(feat_te)} rows")

## LazyPredict Benchmark

In [ ]:
if len(feat_tr) >= 5 and len(feat_te) >= 1:
    try:
        lr = LazyRegressor(verbose=0, ignore_warnings=True, predictions=True)
        lz_m, lz_p = lr.fit(feat_tr[FEAT_COLS], feat_te[FEAT_COLS],
                             feat_tr["y"], feat_te["y"])
        print("LazyPredict top 10 (test):")
        print(lz_m.head(10).to_string())
        best_lz = lz_m.index[0]
        lz_pred = lz_p[best_lz]
        results.append(evaluate(agg_test[:len(lz_pred)], lz_pred, f"LazyPredict-{best_lz}"))
    except Exception as e:
        print(f"LazyPredict failed: {e}"); lz_m = None
else:
    print("Too few rows."); lz_m = None

## FLAML AutoML

In [ ]:
X_tv = feat_tr[FEAT_COLS]; y_tv = feat_tr["y"]
flaml_m = AutoML()
flaml_m.fit(X_tv, y_tv, task="regression", time_budget=FLAML_BUDGET,
            metric="mae", verbose=0, seed=RANDOM_STATE)
if len(feat_te) > 0:
    flaml_pred = flaml_m.predict(feat_te[FEAT_COLS])
    results.append(evaluate(agg_test[:len(flaml_pred)], flaml_pred,
                             f"FLAML ({flaml_m.best_estimator})"))
    print(f"FLAML best: {flaml_m.best_estimator}")

## StatsForecast on Aggregate Series

In [ ]:
sf_traindf = pd.DataFrame({
    "unique_id": "wiki_top100_en",
    "ds": list(agg_train.index),
    "y":  list(agg_train.values),
})
sf = StatsForecast(
    models=[
        AutoARIMA(season_length=52, alias="AutoARIMA"),
        AutoETS(season_length=52, alias="AutoETS"),
        AutoTheta(season_length=52, alias="AutoTheta"),
        SeasonalNaive(season_length=52, alias="SeasonalNaive52"),
        WindowAverage(window_size=4, alias="WindowAvg4"),
    ],
    freq=FREQ, n_jobs=1,
)
print("Fitting StatsForecast on aggregate series...")
sf.fit(sf_traindf)
sf_fcst = sf.predict(h=TEST_WEEKS, level=[80])
print(sf_fcst.to_string())

In [ ]:
print("StatsForecast results:")
for col in ["AutoARIMA", "AutoETS", "AutoTheta", "SeasonalNaive52", "WindowAvg4"]:
    if col in sf_fcst.columns:
        results.append(evaluate(agg_test, sf_fcst[col].values, f"StatsForecast-{col}"))

## StatsForecast — Per-Page Panel Forecasting

In [ ]:
# For top 20 pages individually
top20_ids = (wk_train.groupby("unique_id")["y"].mean()
             .nlargest(20).index.tolist())

panel_train = wk_train[wk_train["unique_id"].isin(top20_ids)][["unique_id","ds","y"]].copy()

# Ensure all series have enough history
obs_counts = panel_train.groupby("unique_id")["ds"].count()
valid_ids  = obs_counts[obs_counts >= 10].index.tolist()
panel_train = panel_train[panel_train["unique_id"].isin(valid_ids)].copy()
print(f"Per-page panel: {len(valid_ids)} pages, {len(panel_train):,} rows")

sf_panel = StatsForecast(
    models=[
        AutoETS(season_length=52, alias="AutoETS"),
        SeasonalNaive(season_length=52, alias="SeasonalNaive52"),
    ],
    freq=FREQ, n_jobs=1,
)
print("Fitting per-page panel...")
sf_panel.fit(panel_train)
panel_fcst = sf_panel.predict(h=TEST_WEEKS)
print(f"Per-page forecasts: {len(panel_fcst)} rows")
print(panel_fcst.head(4).to_string())

In [ ]:
# ── Evaluate per-page aggregate vs actual ────────────────────────────
panel_test = wk_test[wk_test["unique_id"].isin(valid_ids)][["unique_id","ds","y"]]

for col in ["AutoETS", "SeasonalNaive52"]:
    if col in panel_fcst.columns:
        merged = panel_test.merge(panel_fcst[["unique_id","ds",col]], on=["unique_id","ds"], how="inner")
        if len(merged) > 0:
            smv = smape(merged["y"].values, merged[col].values)
            mae_v = mean_absolute_error(merged["y"], merged[col].clip(lower=0))
            print(f"Per-page {col}: SMAPE={smv:.1f}%  MAE={mae_v:.0f}")

## Forecast Visualisation

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(agg_train.index), y=list(agg_train.values),
                          name="Train", mode="lines", line=dict(color="#2563EB")))
actual_ds   = total_weekly[total_weekly["ds"].isin(test_weeks_list)]["ds"]
fig.add_trace(go.Scatter(x=list(actual_ds), y=list(agg_test),
                          name="Actual (test)", mode="lines+markers",
                          line=dict(color="#1E3A5F", dash="dot")))

for col, clr in [("AutoARIMA","#EF4444"),("AutoETS","#F59E0B"),("AutoTheta","#10B981")]:
    if col in sf_fcst.columns:
        fig.add_trace(go.Scatter(x=list(sf_fcst["ds"]), y=list(sf_fcst[col]),
                                  name=f"StatsForecast-{col}",
                                  mode="lines+markers", line=dict(dash="dash",color=clr)))

fig.update_layout(title=f"Aggregate Web Traffic Forecast — StatsForecast (Top {N_TOP_PAGES} EN Pages)",
                  xaxis_title="Week", yaxis_title="Total Weekly Page Views",
                  template="plotly_white")
fig.show()

## Top 3 Models

In [ ]:
results_df = pd.DataFrame(results).sort_values("SMAPE").reset_index(drop=True)
print("=" * 72)
print("ALL MODELS — ranked by SMAPE (competition metric)")
print("=" * 72)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\n{'TOP 3':^72}")
print("=" * 72)
print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="SMAPE", color="SMAPE",
             color_continuous_scale="RdYlGn_r",
             title="Model Comparison — SMAPE (lower = better, competition metric)",
             template="plotly_white")
fig.update_layout(xaxis_tickangle=-40)
fig.show()

## Final Forecast: 8-Week Future Projection

In [ ]:
# Retrain best StatsForecast model on all data
best_sf_col = None
for col in ["AutoARIMA","AutoETS","AutoTheta"]:
    if col in results_df["model"].values[0]:
        best_sf_col = col; break
if best_sf_col is None:
    best_sf_col = "AutoETS"

sf_all_data = pd.DataFrame({
    "unique_id": "wiki_top100_en",
    "ds": list(total_weekly["ds"]),
    "y":  list(total_weekly["y"]),
})

sf_final = StatsForecast(
    models=[eval(f"{best_sf_col}(season_length=52, alias='{best_sf_col}')")],
    freq=FREQ, n_jobs=1
)
sf_final.fit(sf_all_data)
future = sf_final.predict(h=HORIZON, level=[80, 95])
print(f"Future {HORIZON}-week forecast ({best_sf_col}):")
print(future[["ds", best_sf_col]].to_string(index=False))

## Error Analysis

In [ ]:
best_name = results_df.iloc[0]["model"]
print(f"Error analysis — {best_name}  (test set):\n")

for col in ["AutoARIMA","AutoETS","AutoTheta","SeasonalNaive52"]:
    if col in best_name and col in sf_fcst.columns:
        preds = sf_fcst[col].values
        errors = agg_test[:len(preds)] - preds
        errors_pct = errors / np.maximum(agg_test[:len(preds)], 1) * 100

        print(f"Mean error : {errors.mean():+,.0f} (positive = under-prediction)")
        print(f"Max over   : {errors.min():+,.0f}")
        print(f"Max under  : {errors.max():+,.0f}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].bar(range(len(errors)), errors,
                    color=["#EF4444" if e < 0 else "#10B981" for e in errors])
        axes[0].axhline(0, color="black", lw=1)
        axes[0].set_title(f"{col}: Error by Week")
        axes[0].set_xlabel("Test Week")
        axes[0].set_ylabel("Error (views)")

        axes[1].scatter(agg_test[:len(preds)]/1e6, preds/1e6, alpha=0.8, s=40, color="#2563EB")
        lim = max(agg_test[:len(preds)].max(), preds.max()) / 1e6
        axes[1].plot([0, lim], [0, lim], "r--", lw=1)
        axes[1].set_title("Actual vs Predicted (M views)")
        axes[1].set_xlabel("Actual (M views)")
        axes[1].set_ylabel("Predicted (M views)")
        plt.tight_layout(); plt.show()
        break

## Interpretation & Insights

### Why Traffic Spikes Are Hard to Forecast
Wikipedia article traffic spikes are primarily driven by **exogenous news events** — a celebrity death, a sports final, a political event. These are fundamentally unpredictable from historical traffic patterns alone.

**Practical implication**: for CDN planning, it's better to provision for the 95th percentile of historical traffic rather than the point forecast. StatsForecast's `level=[95]` prediction intervals are more useful than point predictions for this use case.

### Weekly Seasonality
Most Wikipedia articles show a clear **Monday peak and Sunday trough**. This is more pronounced for work-related topics (programming, science, finance) and less for entertainment topics (films, celebrities).

### Aggregate vs. Per-Page Forecasting
Forecasting aggregate traffic is significantly easier than per-page forecasting because:
1. Aggregate traffic is smoother (individual spikes average out)
2. Total views have stronger autocorrelation and clearer trend/seasonality
3. Individual pages have high variance and frequent structural breaks

## Limitations

1. **No news event features**: exogenous shocks (celebrity deaths, breaking news) cause massive spikes that historical data cannot predict
2. **Top 100 pages bias**: the most popular articles have different traffic patterns than the average article (power law distribution)
3. **Sample instead of full 145K pages**: the full competition required forecasting all pages — per-page accuracy varies dramatically
4. **Weekly aggregation loses intra-week patterns**: daily resolution would capture day-of-week effects better
5. **Limited history**: ~79 weeks provides less than 2 full annual cycles for SARIMA(52) estimation

## How to Improve This Project

1. **Wikipedia event calendar**: create a CSV of major Wikipedia-relevant events (elections, sports finals, film releases) and add as binary regressors
2. **Per-page clustering**: cluster pages by traffic pattern type (spiky news, stable evergreen, seasonal) and fit separate models per cluster
3. **Darts with TFT or N-BEATS**: neural models can learn cross-page patterns; use `unique_id` encodings to share information across pages
4. **Log-transform the target** before fitting SARIMA — web traffic is log-normal, and models fit much better in log space
5. **Full 145K page pipeline**: with StatsForecast's parallel processing (`n_jobs=-1`), the full dataset can be modelled in minutes on a multi-core machine

## Production Considerations

1. **Rolling re-training**: retrain nightly on the last 90 days; older data has less relevance for trend-following models
2. **Spike detection**: use a z-score threshold on recent history to flag articles currently "in the news" and fall back to higher-capacity allocations
3. **API-level integration**: Wikimedia's Pageviews API provides real-time data; the forecasting pipeline should ingest from the API, not from a static CSV
4. **Confidence intervals for capacity**: always use 95% PI upper bounds for infrastructure planning, not point estimates
5. **Stagger re-training**: with 145K series, stagger by language edition to avoid training bursts

## Common Mistakes to Avoid

1. **Not filling NaN with 0**: missing days in the competition data mean 0 views (not missing data); filling with forward-fill would overstate traffic
2. **Using MAPE on web traffic**: many articles have near-zero-traffic weeks; SMAPE handles this more gracefully
3. **Forgetting to reshape wide → long**: the competition data is wide (pages × dates); almost all TS libraries expect the long format
4. **Training SARIMA(52) without enough history**: need ≥ 2 × 52 = 104 weeks; clip `season_length` to min(52, len//2)
5. **Ignoring page metadata**: article type (talk page vs. article), language, and access type dramatically affect traffic patterns

## Mini Challenge / Exercises

1. **Cluster Wikipedia pages** by traffic pattern using PCA + k-means on the weekly view matrix and visualize the clusters
2. **Implement log-transform pipeline**: apply `log1p` before fitting StatsForecast, reverse with `expm1` — does SMAPE improve?
3. **German vs. English comparison**: filter `de.wikipedia.org` page views and compare weekly seasonality with English pages
4. **Spike detector**: build a simple model that flags pages with >3× their rolling 4-week average as "news spike" pages
5. **Add Wikimedia event data**: find the 2017 US election (Nov 08, 2016) and measure traffic lift for `United_States_presidential_election,_2016` in the days after

## Final Summary & Key Takeaways

### What We Did
- Downloaded the Wikimedia Web Traffic competition dataset
- Parsed page names to extract language, access type, and agent
- Selected top 100 English all-access pages and reshaped from wide to long format
- Aggregated daily → weekly and analysed traffic patterns, day-of-week effects, and spike structure
- Built baselines and benchmarked LazyPredict / FLAML on lag-feature tabular representation
- Fitted StatsForecast AutoARIMA, AutoETS, AutoTheta on both aggregate and per-page panels
- Selected top 3 models by SMAPE and produced an 8-week future forecast

### Key Takeaways
1. **Aggregate forecasting is more reliable** than per-page forecasting for high-variance web traffic
2. **SMAPE is the right metric** for traffic data — handles near-zero denominator better than MAPE
3. **Exogenous events dominate** individual page traffic — models are best for background/evergreen traffic planning
4. **Wide-to-long reshaping** is the most important preprocessing step for panel forecasting
5. **StatsForecast scales efficiently** to thousands of series with `n_jobs=-1` — excellent for production panel forecasting

---
*Notebook #7 of 50 — Time Series Forecasting Portfolio*
*Dataset: Wikipedia Web Traffic (Kaggle competition) | Library: StatsForecast | Freq: Weekly*